# 8_Construccion_Golden_Dimensiones_v2

## Objetivo
Construir las cuatro dimensiones del modelo dimensional Golden y persistirlas como tablas Delta administradas por Unity Catalog.

**Dimensiones:** `dim_fecha`, `dim_cliente`, `dim_producto` y `dim_almacen`.

**Metodología:** ejecutar sección por sección y celda por celda. No utilizar **Run All**. Validar los resultados antes de avanzar.

**Decisiones de diseño:**
- Las dimensiones de cliente, producto y almacén tendrán claves sustitutas numéricas (`*_key`) y conservarán sus identificadores originales (`*_id`).
- Las claves sustitutas se generan ordenando por la clave natural. Son reproducibles mientras el conjunto de origen no cambie; no implementan historial SCD.
- `dim_fecha` cubrirá desde la fecha mínima hasta la máxima encontradas en Ventas e Inventario.
- Los datos de origen se leen exclusivamente desde las tablas Silver.


## 1. Importaciones

**¿Qué realiza esta sección?** Importa las funciones de PySpark necesarias para transformar columnas, generar rangos de fechas y asignar claves sustitutas mediante funciones de ventana.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 2. Parámetros

**¿Qué realiza esta sección?** Define el catálogo, los esquemas de origen y destino, y la ruta física de Golden dentro del contenedor `datalake`.

Los valores predeterminados corresponden al entorno que venimos utilizando. Si se modifican, deben coincidir con los nombres reales de Unity Catalog y Azure Data Lake Storage.

In [ ]:
dbutils.widgets.text("containerDataLake", "datalake")
dbutils.widgets.text("storageName", "stproyectofinalehs")
dbutils.widgets.text("catalogo", "dbw_proyecto_final_ehs")

containerDataLake = dbutils.widgets.get("containerDataLake")
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

esquemaSilver = "silver"
esquemaGolden = "golden"

ruta_golden = f"abfss://{containerDataLake}@{storageName}.dfs.core.windows.net/golden/"

tabla_clientes_silver = f"{catalogo}.{esquemaSilver}.clientes"
tabla_productos_silver = f"{catalogo}.{esquemaSilver}.productos"
tabla_almacenes_silver = f"{catalogo}.{esquemaSilver}.almacenes"
tabla_ventas_silver = f"{catalogo}.{esquemaSilver}.ventas"
tabla_inventario_silver = f"{catalogo}.{esquemaSilver}.inventario"

tabla_dim_fecha = f"{catalogo}.{esquemaGolden}.dim_fecha"
tabla_dim_cliente = f"{catalogo}.{esquemaGolden}.dim_cliente"
tabla_dim_producto = f"{catalogo}.{esquemaGolden}.dim_producto"
tabla_dim_almacen = f"{catalogo}.{esquemaGolden}.dim_almacen"

print("Catálogo:", catalogo)
print("Origen:", f"{catalogo}.{esquemaSilver}")
print("Destino:", f"{catalogo}.{esquemaGolden}")
print("Ruta Golden:", ruta_golden)

## 3. Lectura desde Silver

**¿Qué realiza esta sección?** Lee las cinco tablas Silver necesarias para construir las dimensiones. No modifica ni vuelve a procesar la capa Silver.

In [ ]:
df_clientes_silver = spark.table(tabla_clientes_silver)
df_productos_silver = spark.table(tabla_productos_silver)
df_almacenes_silver = spark.table(tabla_almacenes_silver)
df_ventas_silver = spark.table(tabla_ventas_silver)
df_inventario_silver = spark.table(tabla_inventario_silver)

print("Clientes Silver:", df_clientes_silver.count())
print("Productos Silver:", df_productos_silver.count())
print("Almacenes Silver:", df_almacenes_silver.count())
print("Ventas Silver:", df_ventas_silver.count())
print("Inventario Silver:", df_inventario_silver.count())

## 4. Diagnóstico de fechas

**¿Qué realiza esta sección?** Obtiene la fecha mínima y máxima de `fecha_venta` y `fecha_corte`. La dimensión calendario cubrirá todo ese intervalo para que ambas tablas de hechos puedan relacionarse con la misma dimensión de fecha.

In [ ]:
rango_ventas = df_ventas_silver.select(
    F.to_date("fecha_venta").alias("fecha")
)

rango_inventario = df_inventario_silver.select(
    F.col("fecha_corte").alias("fecha")
)

rango_fechas = rango_ventas.unionByName(rango_inventario).agg(
    F.min("fecha").alias("fecha_minima"),
    F.max("fecha").alias("fecha_maxima")
).first()

fecha_minima = rango_fechas["fecha_minima"]
fecha_maxima = rango_fechas["fecha_maxima"]

assert fecha_minima is not None and fecha_maxima is not None, (
    "No se pudo determinar el rango de fechas desde las tablas Silver."
)
assert fecha_minima <= fecha_maxima, "El rango de fechas es inválido."

print("Fecha mínima:", fecha_minima)
print("Fecha máxima:", fecha_maxima)

## 5. Construcción de dim_fecha

**¿Qué realiza esta sección?** Genera una fila por cada día del intervalo detectado y calcula atributos de calendario: año, trimestre, mes, día, día de semana y fin de semana.

`fecha_key` tendrá formato numérico `AAAAMMDD` y funcionará como clave de relación con las tablas de hechos.

In [ ]:
df_dim_fecha = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit(str(fecha_minima))),
                F.to_date(F.lit(str(fecha_maxima))),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("fecha")
    )
    .withColumn("fecha_key", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("dia", F.dayofmonth("fecha"))
    .withColumn("dia_semana_numero", F.dayofweek("fecha"))
    .withColumn(
        "nombre_mes",
        F.element_at(
            F.create_map(
                *[x for pair in [
                    (1, "enero"), (2, "febrero"), (3, "marzo"),
                    (4, "abril"), (5, "mayo"), (6, "junio"),
                    (7, "julio"), (8, "agosto"), (9, "septiembre"),
                    (10, "octubre"), (11, "noviembre"), (12, "diciembre")
                ] for x in (F.lit(pair[0]), F.lit(pair[1]))]
            ),
            F.col("mes")
        )
    )
    .withColumn(
        "nombre_dia",
        F.element_at(
            F.create_map(
                *[x for pair in [
                    (1, "domingo"), (2, "lunes"), (3, "martes"),
                    (4, "miércoles"), (5, "jueves"), (6, "viernes"),
                    (7, "sábado")
                ] for x in (F.lit(pair[0]), F.lit(pair[1]))]
            ),
            F.col("dia_semana_numero")
        )
    )
    .withColumn("es_fin_semana", F.col("dia_semana_numero").isin(1, 7))
    .select(
        "fecha_key", "fecha", "anio", "trimestre", "mes",
        "nombre_mes", "dia", "dia_semana_numero", "nombre_dia",
        "es_fin_semana"
    )
)

print("Registros dim_fecha:", df_dim_fecha.count())
display(df_dim_fecha.orderBy("fecha").limit(10))

## 6. Construcción de dim_cliente

**¿Qué realiza esta sección?** Construye la dimensión de clientes desde `silver.clientes`. Asigna `cliente_key` con `row_number()` ordenado por `cliente_id` y conserva el identificador original para trazabilidad.

Esta versión es una dimensión tipo 1 de fotografía actual: no mantiene versiones históricas de cambios en los atributos del cliente.

In [ ]:
assert df_clientes_silver.filter(F.col("cliente_id").isNull()).count() == 0, (
    "Existen cliente_id nulos en Silver."
)
assert df_clientes_silver.groupBy("cliente_id").count().filter(
    F.col("count") > 1
).count() == 0, "Existen cliente_id duplicados en Silver."

ventana_cliente = Window.orderBy("cliente_id")

df_dim_cliente = (
    df_clientes_silver
    .select(
        "cliente_id", "nombre", "tipo_documento", "documento",
        "ciudad", "fecha_registro", "tipo_cliente"
    )
    .withColumn("cliente_key", F.row_number().over(ventana_cliente))
    .select(
        "cliente_key", "cliente_id", "nombre", "tipo_documento",
        "documento", "ciudad", "fecha_registro", "tipo_cliente"
    )
)

print("Registros dim_cliente:", df_dim_cliente.count())
display(df_dim_cliente.orderBy("cliente_key").limit(10))

## 7. Construcción de dim_producto

**¿Qué realiza esta sección?** Construye la dimensión de productos desde `silver.productos`, genera `producto_key` y conserva los atributos descriptivos y comerciales del producto.

El costo y el precio se conservan como atributos de referencia. El costo se utilizará posteriormente para estimar la utilidad de las ventas, documentando que no representa necesariamente el costo histórico de cada transacción.

In [ ]:
assert df_productos_silver.filter(F.col("producto_id").isNull()).count() == 0, (
    "Existen producto_id nulos en Silver."
)
assert df_productos_silver.groupBy("producto_id").count().filter(
    F.col("count") > 1
).count() == 0, "Existen producto_id duplicados en Silver."

ventana_producto = Window.orderBy("producto_id")

df_dim_producto = (
    df_productos_silver
    .select(
        "producto_id", "nombre_producto", "categoria", "marca",
        "costo_unitario", "precio_venta", "es_activo"
    )
    .withColumn("producto_key", F.row_number().over(ventana_producto))
    .select(
        "producto_key", "producto_id", "nombre_producto", "categoria",
        "marca", "costo_unitario", "precio_venta", "es_activo"
    )
)

print("Registros dim_producto:", df_dim_producto.count())
display(df_dim_producto.orderBy("producto_key").limit(10))

## 8. Construcción de dim_almacen

**¿Qué realiza esta sección?** Construye la dimensión de almacenes desde `silver.almacenes`, genera `almacen_key` y conserva el identificador y los atributos geográficos y de estado.

In [ ]:
assert df_almacenes_silver.filter(F.col("almacen_id").isNull()).count() == 0, (
    "Existen almacen_id nulos en Silver."
)
assert df_almacenes_silver.groupBy("almacen_id").count().filter(
    F.col("count") > 1
).count() == 0, "Existen almacen_id duplicados en Silver."

ventana_almacen = Window.orderBy("almacen_id")

df_dim_almacen = (
    df_almacenes_silver
    .select(
        "almacen_id", "nombre_almacen", "ciudad", "region", "es_activo"
    )
    .withColumn("almacen_key", F.row_number().over(ventana_almacen))
    .select(
        "almacen_key", "almacen_id", "nombre_almacen",
        "ciudad", "region", "es_activo"
    )
)

print("Registros dim_almacen:", df_dim_almacen.count())
display(df_dim_almacen.orderBy("almacen_key").limit(10))

## 9. Inspección de dimensiones

**¿Qué realiza esta sección?** Resume los conteos y permite inspeccionar muestras de las cuatro dimensiones antes de escribirlas en el almacenamiento Golden.

In [ ]:
conteos_dimensiones = [
    ("dim_fecha", df_dim_fecha.count()),
    ("dim_cliente", df_dim_cliente.count()),
    ("dim_producto", df_dim_producto.count()),
    ("dim_almacen", df_dim_almacen.count())
]

for nombre, cantidad in conteos_dimensiones:
    print(f"{nombre}: {cantidad:,} registros")

print("Rango calendario:", fecha_minima, "a", fecha_maxima)

## 10. Escritura de tablas Delta

**¿Qué realiza esta sección?** Crea el esquema Golden si no existe y persiste las cuatro dimensiones como tablas Delta registradas en Unity Catalog.

La escritura usa `overwrite`: si se vuelve a ejecutar el notebook, las dimensiones se reconstruyen con los datos actuales de Silver. No se modifica Bronze ni Silver.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{esquemaGolden}")
print(f"Esquema disponible: {catalogo}.{esquemaGolden}")

In [ ]:
(
    df_dim_fecha.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}dim_fecha/")
    .saveAsTable(tabla_dim_fecha)
)
print("Tabla creada:", tabla_dim_fecha)

In [ ]:
(
    df_dim_cliente.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}dim_cliente/")
    .saveAsTable(tabla_dim_cliente)
)
print("Tabla creada:", tabla_dim_cliente)

In [ ]:
(
    df_dim_producto.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}dim_producto/")
    .saveAsTable(tabla_dim_producto)
)
print("Tabla creada:", tabla_dim_producto)

In [ ]:
(
    df_dim_almacen.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", f"{ruta_golden}dim_almacen/")
    .saveAsTable(tabla_dim_almacen)
)
print("Tabla creada:", tabla_dim_almacen)

## 11. Validación final

**¿Qué realiza esta sección?** Vuelve a leer las tablas persistidas desde Unity Catalog y comprueba conteos, unicidad de claves sustitutas, unicidad de claves naturales y ausencia de claves nulas.

In [ ]:
dim_fecha_tabla = spark.table(tabla_dim_fecha)
dim_cliente_tabla = spark.table(tabla_dim_cliente)
dim_producto_tabla = spark.table(tabla_dim_producto)
dim_almacen_tabla = spark.table(tabla_dim_almacen)

validaciones = [
    ("dim_fecha", dim_fecha_tabla, "fecha_key", "fecha_key"),
    ("dim_cliente", dim_cliente_tabla, "cliente_key", "cliente_id"),
    ("dim_producto", dim_producto_tabla, "producto_key", "producto_id"),
    ("dim_almacen", dim_almacen_tabla, "almacen_key", "almacen_id")
]

for nombre, df, clave_sustituta, clave_natural in validaciones:
    total = df.count()
    nulos_key = df.filter(F.col(clave_sustituta).isNull()).count()
    duplicados_key = (
        df.groupBy(clave_sustituta).count()
        .filter(F.col("count") > 1).count()
    )
    nulos_natural = df.filter(F.col(clave_natural).isNull()).count()
    duplicados_natural = (
        df.groupBy(clave_natural).count()
        .filter(F.col("count") > 1).count()
    )

    print(f"--- {nombre} ---")
    print("Registros:", total)
    print("Claves sustitutas nulas:", nulos_key)
    print("Claves sustitutas duplicadas:", duplicados_key)
    print("Claves naturales nulas:", nulos_natural)
    print("Claves naturales duplicadas:", duplicados_natural)

    assert total > 0, f"{nombre} no contiene registros."
    assert nulos_key == 0, f"{nombre} contiene claves sustitutas nulas."
    assert duplicados_key == 0, f"{nombre} contiene claves sustitutas duplicadas."
    assert nulos_natural == 0, f"{nombre} contiene claves naturales nulas."
    assert duplicados_natural == 0, f"{nombre} contiene claves naturales duplicadas."

print("Validación final de dimensiones Golden correcta.")